"""
Explorative Analyse: Waldverlust nach dominantem Treiber (Global Forest Watch / OWID)
======================================================================================

Datenquelle:
    https://ourworldindata.org/grapher/tree-cover-loss
    Direkter CSV-Download:
    https://ourworldindata.org/grapher/tree-cover-loss.csv?v=1&csvType=full&useColumnShortNames=false

Datensatzbeschreibung:
    Jährlicher Waldverlust (in Hektar) pro Land, 2001-2024, aufgeschlüsselt nach
    dominantem Treiber: Permanent Agriculture, Hard Commodities, Shifting Cultivation,
    Logging (Forestry), Wildfire, Settlements & Infrastructure, Other Natural Disturbances.

Nutzung:
    1. CSV von obigem Link herunterladen und im selben Ordner wie dieses Skript speichern
       (oder DATA_PATH unten anpassen).
    2. python analyse_waldverlust.py
"""

import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "tree_cover_loss.csv"  # ggf. Pfad anpassen

# ---------------------------------------------------------------------------
# 1. Daten laden
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nSpalten:")
print(df.columns.tolist())
print("\nErste Zeilen:")
print(df.head())

# OWID liefert i.d.R. Spalten: Entity, Code, Year, plus eine Spalte je Treiber
# (Name je nach Exportvariante z.B. "tree_cover_loss_ha__category_permanent_agriculture").
# Wir identifizieren die Treiber-Spalten automatisch:
meta_cols = [c for c in df.columns if c.lower() in ("entity", "code", "year", "country")]
driver_cols = [c for c in df.columns if c not in meta_cols]

print(f"\nErkannte Meta-Spalten: {meta_cols}")
print(f"Erkannte Treiber-Spalten: {driver_cols}")

entity_col = "Entity" if "Entity" in df.columns else "Country"
year_col = "Year" if "Year" in df.columns else "year"

# ---------------------------------------------------------------------------
# 2. Datenqualität prüfen
# ---------------------------------------------------------------------------
print("\nFehlende Werte pro Spalte:")
print(df[driver_cols].isna().sum())

# Kontinente/Aggregate (z.B. "Africa", "World") von einzelnen Ländern trennen,
# damit Ranking-Analysen nicht verzerrt werden. Liste ggf. erweitern.
aggregates = ["World", "Africa", "Asia", "Europe", "North America",
              "South America", "Oceania", "European Union (27)"]
df_countries = df[~df[entity_col].isin(aggregates)].copy()

# ---------------------------------------------------------------------------
# 3. Globaler Trend über Zeit je Treiber
# ---------------------------------------------------------------------------
global_by_year = df[df[entity_col] == "World"].groupby(year_col)[driver_cols].sum()
if global_by_year.empty:
    # Falls kein "World"-Aggregat vorhanden ist: über alle Länder aufsummieren
    global_by_year = df_countries.groupby(year_col)[driver_cols].sum()

fig, ax = plt.subplots(figsize=(10, 6))
global_by_year.plot.area(ax=ax, cmap="tab20")
ax.set_title("Globaler Waldverlust nach dominantem Treiber (2001-2024)")
ax.set_xlabel("Jahr")
ax.set_ylabel("Waldverlust (ha)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
plt.tight_layout()
plt.savefig("waldverlust_global_zeitverlauf.png", dpi=150)
print("\nGespeichert: waldverlust_global_zeitverlauf.png")

# ---------------------------------------------------------------------------
# 4. Top 10 Länder nach kumuliertem Gesamtverlust
# ---------------------------------------------------------------------------
df_countries["total_loss"] = df_countries[driver_cols].sum(axis=1)
totals_by_country = df_countries.groupby(entity_col)["total_loss"].sum().sort_values(ascending=False)
top10 = totals_by_country.head(10)

print("\nTop 10 Länder nach kumuliertem Waldverlust (2001-2024):")
print(top10)

# Anteil der Treiber innerhalb dieser Top-10-Länder (Balkendiagramm, gestapelt)
top10_detail = (
    df_countries[df_countries[entity_col].isin(top10.index)]
    .groupby(entity_col)[driver_cols].sum()
    .loc[top10.index]  # Reihenfolge nach Gesamtverlust beibehalten
)
top10_share = top10_detail.div(top10_detail.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 6))
top10_share.plot.barh(stacked=True, ax=ax, cmap="tab20")
ax.set_title("Anteil der Treiber am Waldverlust – Top 10 Länder")
ax.set_xlabel("Anteil am Gesamtverlust")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
plt.tight_layout()
plt.savefig("waldverlust_top10_treiberanteil.png", dpi=150)
print("Gespeichert: waldverlust_top10_treiberanteil.png")

# ---------------------------------------------------------------------------
# 5. Welcher Treiber dominiert weltweit am häufigsten?
# ---------------------------------------------------------------------------
dominant_driver_per_country = df_countries.groupby(entity_col)[driver_cols].sum().idxmax(axis=1)
print("\nHäufigster dominanter Treiber über alle Länder hinweg:")
print(dominant_driver_per_country.value_counts())

print("\nFertig. Passe DATA_PATH an, falls die CSV woanders liegt.")

In [3]:
 
import pandas as pd
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [4]:
DATA_PATH = "drivers-deforestation-commodity.csv"

In [5]:
df = pd.read_csv(DATA_PATH)
 
print("Shape:", df.shape)
print("\nSpalten:")
print(df.columns.tolist())
print("\nErste Zeilen:")
print(df.head())

Shape: (4001, 11)

Spalten:
['Entity', 'Code', 'Year', 'Beef', 'Forestry (paper and wood)', 'Palm oil', 'Soy', 'Cocoa and coffee', 'Cereals', 'Roots and tubers', 'Other crops']

Erste Zeilen:
        Entity Code  Year  Beef  Forestry (paper and wood)  Palm oil  Soy  \
0  Afghanistan  AFG  2001  2.95                        NaN       NaN  NaN   
1  Afghanistan  AFG  2002  6.85                        NaN       NaN  NaN   
2  Afghanistan  AFG  2003  2.99                        NaN       NaN  NaN   
3  Afghanistan  AFG  2004  3.20                        NaN       NaN  NaN   
4  Afghanistan  AFG  2005  7.29                        NaN       NaN  NaN   

   Cocoa and coffee  Cereals  Roots and tubers  Other crops  
0               NaN    63.33              0.33         3.52  
1               NaN    39.93              0.30         3.93  
2               NaN     6.83               NaN         3.38  
3               NaN    24.37              0.13         4.00  
4               NaN     2.94       

In [6]:
df.dtypes

Entity                           str
Code                             str
Year                           int64
Beef                         float64
Forestry (paper and wood)    float64
Palm oil                     float64
Soy                          float64
Cocoa and coffee             float64
Cereals                      float64
Roots and tubers             float64
Other crops                  float64
dtype: object

In [9]:
df["Entity"].unique()

<StringArray>
[        'Afghanistan',              'Africa',             'Albania',
             'Algeria',              'Angola', 'Antigua and Barbuda',
           'Argentina',             'Armenia',                'Asia',
           'Australia',
 ...
       'United States',             'Uruguay',          'Uzbekistan',
             'Vanuatu',           'Venezuela',             'Vietnam',
               'World',               'Yemen',              'Zambia',
            'Zimbabwe']
Length: 192, dtype: str